# Kafka scoring (Module 4, streaming)

Kafka runs on the cluster's master (installed at cluster start by `infra/kafka_init.sh`). The **scorer** (Spark Structured Streaming) reads the topic and scores each rating with the saved model named by `streaming.model`; the **producer** replays test ratings into Kafka at `streaming.replay_rate_per_sec`. Results: `results/streaming_latency.csv`, `results/streaming_summary.json`, per-batch predictions under `results/streaming/`.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls streaming infra/kafka_init.sh scripts/run_streaming.py config.yaml

In [ ]:
# Inputs: ID mappings + test set (Module 1) and the saved models (FinalModel.ipynb)
!gcloud storage ls gs://YOUR_BUCKET/full/processed/
!gcloud storage ls gs://YOUR_BUCKET/full/results/models/

In [ ]:
# Choose the model to score with (written into config.yaml, the single source of settings)
MODEL = "centralized"     # or "federated" (simulated on-device scoring, see SPEC.md)
!sed -i 's/^  model: .*/  model: {MODEL}/' config.yaml
!grep -E "^  (model|replay_rate_per_sec|replay_max_events|kafka_bootstrap_servers|checkpoint_location):" config.yaml

In [ ]:
# Fresh run: remove the previous run's checkpoint and streaming outputs (a new cluster has a new,
# empty Kafka, so an old checkpoint would point at message offsets that no longer exist)
import yaml
cfg = yaml.safe_load(open("config.yaml")); I, S, D = cfg["infra"], cfg["streaming"], cfg["data"]
RES = D["results_prefix"].rstrip("/")
!gcloud storage rm -r {S['checkpoint_location']} {RES}/streaming {RES}/streaming_latency {RES}/streaming_latency.csv {RES}/streaming_summary.json 2>/dev/null; echo cleared

In [ ]:
# ONE-TIME SETUP (runs in Colab, which has internet): the cluster cannot reach the internet, so copy
# Kafka and Spark's Kafka connector into the bucket once; the cluster then fetches them from there.
import yaml, glob, os
S0 = yaml.safe_load(open("config.yaml"))["streaming"]
KAFKA_TGZ = S0["kafka_download_url"]                  # gs://.../kafka_2.13-3.7.1.tgz
JARS = S0["spark_kafka_jars"].rstrip("/") + "/"       # gs://.../spark-kafka-jars/
# 1) Kafka broker (from the Apache archive -> bucket)
!curl -fsSL https://archive.apache.org/dist/kafka/3.7.1/kafka_2.13-3.7.1.tgz -o /tmp/kafka.tgz && gcloud storage cp /tmp/kafka.tgz {KAFKA_TGZ}
# 2) Spark 3.5.3 Kafka connector + its dependencies, resolved by Spark itself (same set that
#    spark.jars.packages would download), then uploaded
!pip -q install pyspark==3.5.3
from pyspark.sql import SparkSession
_s = (SparkSession.builder.master("local[1]")
      .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3").getOrCreate()); _s.stop()
jars = glob.glob(os.path.expanduser("~/.ivy2/jars/*.jar")); print(len(jars), "jars:", [os.path.basename(j) for j in jars])
!gcloud storage cp ~/.ivy2/jars/*.jar {JARS}
!gcloud storage ls -l {KAFKA_TGZ} {JARS}

In [ ]:
# Cluster with Kafka on the master (KAFKA=1). 4 workers: the scorer keeps the model and the
# ID mapping tables (millions of users) in memory.
WORKERS = 4
!KAFKA=1 CONFIG=config.yaml bash infra/dataproc_launch.sh {WORKERS}

In [ ]:
# Package the code, upload the config, define job settings
import shutil, subprocess, time
stage = "/tmp/stage"; shutil.rmtree(stage, ignore_errors=True)
for pkg in ["etl", "centralized", "evaluation", "federated", "benchmarks", "streaming"]:
    shutil.copytree(pkg, f"{stage}/{pkg}", ignore=shutil.ignore_patterns("__pycache__"))
shutil.make_archive("/tmp/code", "zip", stage)
CFG = RES.rsplit("/", 1)[0] + "/config.yaml"
!gcloud storage cp config.yaml {CFG}
n = WORKERS * int(I["executors_per_node"])
P = (f"spark.dynamicAllocation.enabled=false,spark.executor.instances={n},spark.executor.cores={I['executor_cores']},"
     f"spark.executor.memory={I['executor_memory']},spark.executor.memoryOverhead={I['executor_memory_overhead']},"
     f"spark.driver.memory={I['driver_memory']}")
JOB = f"gcloud dataproc jobs submit pyspark scripts/run_streaming.py --cluster={I['cluster_name']} --region={I['region']} --py-files=/tmp/code.zip"
# the Kafka connector JARs from the bucket (the cluster cannot download them itself)
# Only the 4 JARs Dataproc's Spark lacks; the resolver also pulled Hadoop/slf4j/lz4/snappy copies that
# Dataproc already has in other versions, and duplicating those risks classpath conflicts.
NEEDED = ("spark-sql-kafka-0-10", "spark-token-provider-kafka-0-10", "kafka-clients", "commons-pool2")
KJARS = ",".join(j for j in subprocess.run(f"gcloud storage ls {S['spark_kafka_jars'].rstrip('/')}/*.jar", shell=True,
                                           capture_output=True, text=True).stdout.split()
                 if any(k in j for k in NEEDED))
# The scorer holds one large executor per worker for the whole run, so the producer and summary
# jobs get ONE small executor that fits beside it (otherwise they wait forever for resources).
P_SMALL = ("spark.dynamicAllocation.enabled=false,spark.executor.instances=1,spark.executor.cores=1,"
           "spark.executor.memory=2g,spark.executor.memoryOverhead=1g")
print(CFG); print(P); print(len(KJARS.split(',')), 'connector jars')

In [ ]:
# Start the SCORER in the background (it waits for events; stops after idle_stop_seconds of silence)
out = subprocess.run(f"{JOB} --async --format='value(reference.jobId)' "
                     f"--properties={P} --jars={KJARS} -- --config={CFG} --action=scorer",
                     shell=True, capture_output=True, text=True)
SCORER_ID = out.stdout.strip().splitlines()[-1]
print("scorer job:", SCORER_ID)
time.sleep(120)   # give it time to start, fetch the Kafka connector and cache the model
!gcloud dataproc jobs describe {SCORER_ID} --region={I['region']} --format="value(status.state)"

In [ ]:
# PRODUCER: replay test ratings into Kafka with Spark's Kafka writer (blocking; ~replay_max_events /
# replay_rate_per_sec seconds). One small executor beside the scorer; needs the connector JARs too.
!{JOB} --properties={P_SMALL} --jars={KJARS} -- --config={CFG} --action=producer 2>&1 | grep --line-buffered -E "^\[producer|Traceback|Error|state:"

In [ ]:
# Wait for the scorer to drain the topic and stop on its own, then show its log
!gcloud dataproc jobs wait {SCORER_ID} --region={I['region']} 2>&1 | grep --line-buffered -E "^\[scorer|Traceback|Error|state:" | tail -25

In [ ]:
# SUMMARY: latency percentiles, throughput, unknown IDs, duplicates, RMSE of streamed predictions
!{JOB} --properties={P_SMALL} -- --config={CFG} --action=summarize 2>&1 | grep --line-buffered -E "^\[scorer|Traceback|Error|state:"
!gcloud storage cat {RES}/streaming_summary.json

In [ ]:
# OPTIONAL restart check: rerun the scorer on the same checkpoint. Expect no new batches and,
# after summarizing again, duplicate_event_ids = 0.
!{JOB} --properties={P} --jars={KJARS} -- --config={CFG} --action=scorer --max-seconds 120 2>&1 | grep -E "^\[scorer|Traceback|Error|state:"
!{JOB} --properties={P_SMALL} -- --config={CFG} --action=summarize 2>&1 | grep --line-buffered -E "^\[scorer.*summary"


In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh